[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/41_maxpool2d_solution.ipynb)

# ✅ Solution: 2D Max Pooling

**2D Max Pooling** を実装する。Conv2d (#22) の自然な相方で、ほぼ全 CNN block の後に 来る down-sampling layer。

$$\text{out}[b, c, i, j] = \max_{(p,q) \in \text{window}(i,j)} x[b, c, p, q]$$

> 💡 **どこで使う**：Conv 後の標準的な down-sampling。最大値を残して空間サイズを半分にする。

<details>
<summary>📖 もっと詳しく</summary>

VGG 系で頻出。最近のアーキテクチャ (ResNet 後期、ViT) は stride 付き conv で代替する設計も多い。

落とし穴: padding は `-inf` で（0 だと負入力で誤る）、勾配は argmax 位置にしか流れない。

</details>

### AvgPool との違い
- max は **non-smooth**（勾配は argmax 位置にだけ流れる）
- Padding は **`-inf`** で行う — 0 だと負入力で誤る


### Signature
```python
def my_max_pool2d(x, kernel_size, stride=None, padding=0):
    # x: (B, C, H, W)
    # Returns: (B, C, H_out, W_out)
```

### Rules
- `F.max_pool2d`, `nn.MaxPool2d`, `F.adaptive_max_pool2d` は **使わない**
- `unfold`, `reshape`, `torch.max`/`amax` は許可
- `stride` の default は `kernel_size` (non-overlapping windows)
- `padding > 0` の場合、`float('-inf')` で pad — 0 だと negative input が pad に dominate される
- 勾配は各 window の argmax 位置にのみ流れる

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✅ SOLUTION

def my_max_pool2d(x, kernel_size, stride=None, padding=0):
    if stride is None:
        stride = kernel_size
    if padding > 0:
        x = F.pad(x, [padding] * 4, value=float('-inf'))
    # Sliding window 抽出: (B, C, H_out, W_out, kH, kW)
    patches = x.unfold(2, kernel_size, stride).unfold(3, kernel_size, stride)
    # window 次元で max
    return patches.amax(dim=(-1, -2))

In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)

x = torch.randn(1, 1, 4, 4)
print('Input:\n', x.squeeze())
print('Output:\n', my_max_pool2d(x, kernel_size=2).squeeze())
print('Ref:\n', F.max_pool2d(x, kernel_size=2).squeeze())

In [ ]:
from torch_judge import check
check("maxpool2d")